In [1]:
#
# ⚡ UNIVERSAL FIRST CELL - Run this FIRST in every notebook!
# Compatible with: Google Colab, GitHub Codespaces, Local
#

import os
import subprocess
import sys

# Detect environment
IS_COLAB = "google.colab" in sys.modules
IS_CODESPACES = os.path.exists("/.devcontainer") or os.path.exists("/workspaces")
IS_LOCAL = not (IS_COLAB or IS_CODESPACES)

print(f"🚀 Environment: {'Colab' if IS_COLAB else 'Codespaces' if IS_CODESPACES else 'Local'}")

# Clone repo in Colab if needed
if IS_COLAB and not os.path.exists("computer-data-analysis-report"):
    print("Cloning repository...")
    !git clone --depth 1 https://github.com/Aidas-dev/computer-data-analysis-report.git

# Change to repo directory
repo_dir = "computer-data-analysis-report"
if os.path.exists(repo_dir):
    %cd {repo_dir}
    !git pull -q

# Install uv (faster than pip)
!pip install uv -q

# Install dependencies with uv
!uv pip install --system -r requirements.txt -q

# Setup DVC in Colab
if IS_COLAB:
    from google.colab import userdata
    try:
        access_key = userdata.get("OCI_ACCESS_KEY")
        secret_key = userdata.get("OCI_SECRET_KEY")
        !dvc remote modify --local oracle_remote access_key_id {access_key}
        !dvc remote modify --local oracle_remote secret_access_key {secret_key}
        !dvc pull -q
    except:
        pass

# Ensure we are in the root directory for relative paths
while not os.path.exists('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print(f"📁 Working directory set to: {os.getcwd()}")

print("✅ Environment ready!")


🚀 Environment: Local


error: File not found: `requirements.txt`


📁 Working directory set to: /home/aidmantas/repos/computer-data-analysis-report
✅ Environment ready!


# Data Merging: Unified AI Data Center Dataset

This notebook combines multiple raw data sources into a single, clean feature matrix for machine learning model training.

### Sources:
1. **Buildout Promises** (`data/raw/buildout_promises_expanded.csv`): The foundation (target variables).
2. **Company Financials** (`data/raw/company_financials_expanded.csv`): Stock performance and ratios.
3. **Grid Queues** (`data/raw/grid_interconnection_queue.csv`): Regional infrastructure constraints.
4. **Macro Economic Indicators** (`data/raw/macro_economic_indicators.csv`): Global economic climate.

In [2]:
import pandas as pd
import numpy as np

# Load data
promises = pd.read_csv('data/raw/buildout_promises_expanded.csv')
financials = pd.read_csv('data/raw/company_financials_expanded.csv')
grid = pd.read_csv('data/raw/grid_interconnection_queue.csv')
macro = pd.read_csv('data/raw/macro_economic_indicators.csv')

# Convert dates
promises['announcement_date'] = pd.to_datetime(promises['announcement_date'])
promises['target_date'] = pd.to_datetime(promises['target_date'])
# Macro dates often have timezone info from APIs (FRED/ISO8601)
macro['date'] = pd.to_datetime(macro['date'], format='ISO8601', utc=True).dt.tz_localize(None)
grid['Queue Date'] = pd.to_datetime(grid['Queue Date'], errors='coerce')

print(f"Loaded {len(promises)} promises, {len(financials)} financial rows, {len(grid)} grid projects, {len(macro)} macro observations.")

Loaded 34 promises, 10000 financial rows, 6043 grid projects, 1665 macro observations.


### 1. Macro Economic Merging (Quarterly Resampling)
We resample the macro data to quarterly frequency, taking the most recent available value in the quarter.

In [3]:
# Resample macro to Quarterly (end of quarter)
# Handles compatibility between QE (new pandas) and Q (old pandas)
try:
    macro_q = macro.set_index('date').resample('QE').last().reset_index()
except:
    macro_q = macro.set_index('date').resample('Q').last().reset_index()

macro_q['quarter'] = macro_q['date'].dt.to_period('Q')

# Add quarter to promises for merging
promises['announcement_quarter'] = promises['announcement_date'].dt.to_period('Q')

# Merge
merged_df = promises.merge(macro_q.drop(columns=['date']), left_on='announcement_quarter', right_on='quarter', how='left')
merged_df = merged_df.drop(columns=['quarter'])

print(f"After Macro Merge: {merged_df.shape}")
print(merged_df[['company', 'announcement_date', 'Real GDP', 'Unemployment Rate']].head())

After Macro Merge: (34, 22)
     company announcement_date   Real GDP  Unemployment Rate
0  Microsoft        2021-03-01  21082.134                6.1
1  Microsoft        2022-06-15  21967.045                3.6
2  Microsoft        2022-11-01  22278.345                3.5
3  Microsoft        2024-01-15  23082.119                3.9
4     Google        2021-09-01  21617.828                4.7


### 2. Grid Interconnection Queue Features
We calculate the "Queue Pressure" per ISO/Quarter (number of projects in queue).

In [4]:
# Aggregate grid projects by ISO and Quarter
grid['quarter'] = grid['Queue Date'].dt.to_period('Q')
grid_counts = grid.groupby(['iso', 'quarter']).size().reset_index(name='iso_queue_depth')

# Merge with main dataset
merged_df = merged_df.merge(grid_counts, left_on=['iso', 'announcement_quarter'], right_on=['iso', 'quarter'], how='left')
merged_df['iso_queue_depth'] = merged_df['iso_queue_depth'].fillna(0)
merged_df = merged_df.drop(columns=['quarter'])

print(f"After Grid Merge: {merged_df.shape}")

After Grid Merge: (34, 23)


### 3. Save Final Dataset
We save the processed features for modeling.

In [5]:
os.makedirs('data/processed', exist_ok=True)
merged_df.to_csv('data/processed/dataset_for_ml.csv', index=False)
print(f"🎉 Final dataset saved: data/processed/dataset_for_ml.csv ({len(merged_df)} rows)")

🎉 Final dataset saved: data/processed/dataset_for_ml.csv (34 rows)
